# Modulo 9 · Lezione 3 — Archiviazione SQL

**Progetto Finale: Sistema Intelligente di Predizione ed Esposizione di Modelli**

---

## Cosa vedremo oggi?

| # | Argomento |
|---|------|
| 1 | SQLite con `sqlite3` — SQL puro |
| 2 | SQLAlchemy — ORM di Python |
| 3 | Caricare i dati elaborati nel DB |
| 4 | Query analitiche con SQL |
| 5 | Integrazione pandas ↔ SQL |

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

DB_PATH = 'data/db/vinos.db'
Path('data/db').mkdir(parents=True, exist_ok=True)

# Caricare i dati elaborati
df = pd.read_parquet('data/processed/vinos_procesado.parquet')
print(f"Dataset caricato: {df.shape}")
df.head(3)

---
## 1. SQLite con `sqlite3`

`sqlite3` è la libreria standard di Python per i database SQL. Non devi installare nulla: SQLite salva l'intero DB in un unico file `.db`.

In [ ]:
# Creare connessione e tabella
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# DROP + CREATE per iniziare puliti
cursor.execute("DROP TABLE IF EXISTS vinos")

cursor.execute("""
CREATE TABLE vinos (
    id                     INTEGER PRIMARY KEY AUTOINCREMENT,
    fixed_acidity          REAL,
    volatile_acidity       REAL,
    citric_acid            REAL,
    residual_sugar         REAL,
    chlorides              REAL,
    free_sulfur_dioxide    REAL,
    total_sulfur_dioxide   REAL,
    density                REAL,
    pH                     REAL,
    sulphates              REAL,
    alcohol                REAL,
    so2_ratio              REAL,
    acid_index             REAL,
    density_diff           REAL,
    tipo_encoded           INTEGER,
    quality                INTEGER,
    buen_vino              INTEGER,
    tipo                   TEXT
)
""")

conn.commit()
print("✓ Tabella 'vinos' creata in", DB_PATH)

In [ ]:
# Inserire i dati riga per riga (per capire il meccanismo)
# In produzione useremmo executemany o to_sql

COLUMNAS = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol', 'so2_ratio', 'acid_index', 'density_diff',
    'tipo_encoded', 'quality', 'buen_vino', 'tipo'
]

# Inserire le prime 10 righe manualmente per vedere il meccanismo
placeholders = ', '.join(['?'] * len(COLUMNAS))
sql_insert = f"INSERT INTO vinos VALUES (NULL, {placeholders})"

for _, row in df.head(10).iterrows():
    valores = tuple(row[c] for c in COLUMNAS)
    cursor.execute(sql_insert, valores)

conn.commit()
print("10 righe inserite una a una")

# Verificare
cursor.execute("SELECT COUNT(*) FROM vinos")
print(f"Righe nel DB: {cursor.fetchone()[0]}")

# executemany è PIÙ EFFICIENTE per molte righe
cursor.execute("DELETE FROM vinos")  # pulire per inserire tutto
filas = [tuple(row[c] for c in COLUMNAS) for _, row in df.iterrows()]
cursor.executemany(sql_insert, filas)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM vinos")
print(f"Righe dopo executemany: {cursor.fetchone()[0]}")

In [ ]:
# Query SQL di base

# SELECT di base
cursor.execute("SELECT id, tipo, quality, alcohol FROM vinos LIMIT 5")
print("SELECT di base:")
for fila in cursor.fetchall():
    print(f"  {fila}")

# WHERE + ORDER BY
cursor.execute("""
    SELECT tipo, ROUND(AVG(quality), 2) AS calidad_media, COUNT(*) AS n
    FROM vinos
    GROUP BY tipo
    ORDER BY calidad_media DESC
""")
print("\nQualità media per tipo:")
for fila in cursor.fetchall():
    print(f"  {fila}")

# Subquery: vini con qualità superiore alla media del loro tipo
cursor.execute("""
    SELECT v.tipo, v.quality, v.alcohol
    FROM vinos v
    WHERE v.quality > (
        SELECT AVG(v2.quality) FROM vinos v2 WHERE v2.tipo = v.tipo
    )
    LIMIT 5
""")
print("\nVini sopra la media del loro tipo:")
for fila in cursor.fetchall():
    print(f"  {fila}")

---
## 2. SQLAlchemy — ORM di Python

In [ ]:
from sqlalchemy import create_engine, text, Column, Integer, Float, String
from sqlalchemy.orm import DeclarativeBase, Session

# Creare l'engine (l'URL sqlite:///percorso è relativo alla directory corrente)
engine = create_engine(f'sqlite:///{DB_PATH}', echo=False)

print(f"✓ Engine connesso a {DB_PATH}")
print(f"  Dialect: {engine.dialect.name}")

# Verificare le tabelle esistenti
with engine.connect() as c:
    resultado = c.execute(text("SELECT name FROM sqlite_master WHERE type='table'"))
    tablas = [row[0] for row in resultado]
    print(f"  Tabelle: {tablas}")

In [ ]:
# Definire il modello ORM
class Base(DeclarativeBase):
    pass

class Prediccion(Base):
    """Tabella per salvare le predizioni del modello in produzione."""
    __tablename__ = 'predicciones'

    id              = Column(Integer, primary_key=True, autoincrement=True)
    timestamp       = Column(String)   # ISO 8601
    tipo_vino       = Column(String)
    alcohol         = Column(Float)
    pH              = Column(Float)
    quality_real    = Column(Integer)
    quality_pred    = Column(Float)
    buen_vino_pred  = Column(Integer)
    confianza       = Column(Float)
    modelo_version  = Column(String)

    def __repr__(self):
        return f"<Prediccion id={self.id} pred={self.quality_pred:.1f} real={self.quality_real}>"

# Creare la tabella nel DB
Base.metadata.create_all(engine)
print("✓ Tabella 'predicciones' creata")

In [ ]:
from datetime import datetime
import random

# Inserire predizioni simulate (verranno usate in C5 con il modello reale)
random.seed(42)
predicciones_demo = []
for i, (_, row) in enumerate(df.head(50).iterrows()):
    ruido = random.gauss(0, 0.5)
    pred_q = float(row['quality']) + ruido
    pred_b = int(pred_q >= 7)
    predicciones_demo.append(Prediccion(
        timestamp      = datetime.now().isoformat(),
        tipo_vino      = row['tipo'],
        alcohol        = float(row['alcohol']),
        pH             = float(row['pH']),
        quality_real   = int(row['quality']),
        quality_pred   = round(pred_q, 2),
        buen_vino_pred = pred_b,
        confianza      = round(random.uniform(0.6, 0.95), 2),
        modelo_version = 'v1.0-demo'
    ))

with Session(engine) as session:
    session.add_all(predicciones_demo)
    session.commit()

print(f"✓ {len(predicciones_demo)} predizioni demo inserite")

---
## 3. Integrazione pandas ↔ SQL

In [ ]:
# pd.read_sql + pd.to_sql — i metodi più comodi

# Leggere l'intera tabella
df_desde_bd = pd.read_sql('SELECT * FROM vinos LIMIT 5', engine)
print("Prime righe dal DB:")
print(df_desde_bd[['tipo','quality','alcohol','pH']].to_string(index=False))

# Query analitica direttamente come DataFrame
df_resumen = pd.read_sql("""
    SELECT
        tipo,
        quality,
        COUNT(*)                     AS n_vinos,
        ROUND(AVG(alcohol), 2)       AS alcohol_medio,
        ROUND(AVG(pH), 2)            AS pH_medio,
        ROUND(AVG(sulphates), 3)     AS sulfatos_medios
    FROM vinos
    GROUP BY tipo, quality
    ORDER BY tipo, quality
""", engine)

print("\nRiepilogo per tipo e qualità:")
print(df_resumen.to_string(index=False))

In [ ]:
# to_sql — salvare un DataFrame direttamente in una nuova tabella

df_stats = df.groupby(['tipo', 'quality']).agg(
    n=('quality', 'count'),
    alcohol_medio=('alcohol', 'mean'),
    pH_medio=('pH', 'mean'),
).reset_index().round(3)

df_stats.to_sql('resumen_calidad', engine,
                if_exists='replace',  # replace/append/fail
                index=False)

print("Tabella 'resumen_calidad' salvata nel DB")

# Verificare
df_check = pd.read_sql('SELECT * FROM resumen_calidad', engine)
print(df_check)

---
## 4. Query analitiche avanzate

In [ ]:
import matplotlib.pyplot as plt

# Window functions in SQLite (disponibili dalla v3.25)
df_window = pd.read_sql("""
    SELECT
        tipo,
        quality,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY tipo), 2) AS pct_en_tipo
    FROM vinos
    GROUP BY tipo, quality
    ORDER BY tipo, quality
""", engine)

print("Distribuzione percentuale della qualità per tipo:")
print(df_window.to_string(index=False))

# Visualizzare
fig, ax = plt.subplots(figsize=(10, 5))
for tipo in df_window['tipo'].unique():
    sub = df_window[df_window['tipo'] == tipo]
    ax.plot(sub['quality'], sub['pct_en_tipo'], marker='o', label=tipo, linewidth=2)

ax.set_xlabel('Qualità'); ax.set_ylabel('% all\'interno del tipo')
ax.set_title('Distribuzione della qualità per tipo di vino')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Analizzare le predizioni demo
df_pred = pd.read_sql("""
    SELECT
        tipo_vino,
        COUNT(*)                                          AS n,
        ROUND(AVG(ABS(quality_real - quality_pred)), 3)  AS mae,
        ROUND(AVG(CASE WHEN buen_vino_pred = (quality_real >= 7)
                       THEN 1 ELSE 0 END), 3)            AS accuracy,
        ROUND(AVG(confianza), 3)                         AS confianza_media
    FROM predicciones
    GROUP BY tipo_vino
""", engine)

print("Metriche delle predizioni demo per tipo:")
print(df_pred.to_string(index=False))

# Chiudere la connessione sqlite3 manuale
conn.close()
print("\n✓ Connessione sqlite3 chiusa")

---
## Pratica

### Esercizio 1 — Top 10 vini migliori

**Consegna:** Scrivi una query SQL che restituisca i 10 vini con la qualità più alta.
In caso di parità nella qualità, ordinali per alcol da maggiore a minore.
Mostra: id, tipo, quality, alcohol, pH.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine('sqlite:///data/db/vinos.db')

df_top = pd.read_sql("""
    SELECT id, tipo, quality, ROUND(alcohol, 2) AS alcohol, ROUND(pH, 2) AS pH
    FROM vinos
    ORDER BY quality DESC, alcohol DESC
    LIMIT 10
""", engine)

print("Top 10 vini migliori:")
print(df_top.to_string(index=False))
```
</details>

---
### Esercizio 2 — Correlazione SQL vs pandas

**Consegna:** In pandas calcoliamo le correlazioni con `.corr()`. Anche in SQL puro è possibile.

Formula di Pearson: `r = (n·Σxy - Σx·Σy) / √[(n·Σx² - (Σx)²)·(n·Σy² - (Σy)²)]`

1. Scrivi una query SQL che calcoli la correlazione tra `alcohol` e `quality`
2. Confronta il risultato con quello che dà `df[['alcohol','quality']].corr()`

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine('sqlite:///data/db/vinos.db')

# Correlazione in SQL
with engine.connect() as conn:
    resultado = conn.execute(text("""
        SELECT
            (COUNT(*) * SUM(alcohol * quality) - SUM(alcohol) * SUM(quality)) /
            SQRT(
                (COUNT(*) * SUM(alcohol*alcohol) - SUM(alcohol)*SUM(alcohol)) *
                (COUNT(*) * SUM(quality*quality) - SUM(quality)*SUM(quality))
            ) AS correlacion_pearson
        FROM vinos
    """))
    r_sql = resultado.fetchone()[0]

# Correlazione in pandas
df = pd.read_parquet('data/processed/vinos_procesado.parquet')
r_pandas = df['alcohol'].corr(df['quality'])

print(f"Correlazione alcohol-quality:")
print(f"  SQL:    {r_sql:.6f}")
print(f"  pandas: {r_pandas:.6f}")
print(f"  Differenza: {abs(r_sql - r_pandas):.2e}")
```
</details>

---
### Esercizio 3 — Indice di prestazioni

**Consegna:** Crea un indice sulla colonna `quality` e confronta il tempo di una query con e senza indice:

1. `EXPLAIN QUERY PLAN SELECT ... WHERE quality = 8` prima dell'indice
2. `CREATE INDEX idx_quality ON vinos(quality)`
3. Ripeti l'EXPLAIN e osserva il cambiamento
4. Misura il tempo con `%timeit` in una SELECT con filtro

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import sqlite3

conn = sqlite3.connect('data/db/vinos.db')
cursor = conn.cursor()

# Piano di query SENZA indice
cursor.execute("DROP INDEX IF EXISTS idx_quality")
cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM vinos WHERE quality = 8")
print("Senza indice:", cursor.fetchall())

# Creare l'indice
cursor.execute("CREATE INDEX IF NOT EXISTS idx_quality ON vinos(quality)")
conn.commit()

# Piano di query CON indice
cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM vinos WHERE quality = 8")
print("Con indice:", cursor.fetchall())

conn.close()

# Misurare il tempo
import timeit
def query_sin_indice():
    c = sqlite3.connect('data/db/vinos.db')
    c.execute("SELECT * FROM vinos WHERE quality = 8")
    c.fetchall(); c.close()

t = timeit.timeit(query_sin_indice, number=100)
print(f"\n100 query: {t:.3f}s ({t/100*1000:.2f}ms per query)")
print("Nota: con SQLite in locale la differenza è piccola —"
      " in PostgreSQL con milioni di righe è enorme")
```
</details>

---
### Esercizio 4 — Logger delle predizioni

**Consegna:** Crea una funzione `log_prediccion(engine, datos_entrada, prediccion)` che:

1. Inserisca una nuova riga nella tabella `predicciones`
2. Restituisca l'`id` assegnato
3. Includa un campo `timestamp` con la data/ora attuale
4. Sia sicura in caso di input errati (try/except + rollback)

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
from sqlalchemy import create_engine, text
from datetime import datetime

engine = create_engine('sqlite:///data/db/vinos.db')

def log_prediccion(engine, datos_entrada: dict, prediccion: dict) -> int | None:
    """
    Registra una predizione nel DB e restituisce l'id assegnato.
    datos_entrada: {'tipo_vino', 'alcohol', 'pH'}
    prediccion:    {'quality_real', 'quality_pred', 'buen_vino_pred', 'confianza', 'modelo_version'}
    """
    sql = text("""
        INSERT INTO predicciones
            (timestamp, tipo_vino, alcohol, pH,
             quality_real, quality_pred, buen_vino_pred, confianza, modelo_version)
        VALUES
            (:timestamp, :tipo_vino, :alcohol, :pH,
             :quality_real, :quality_pred, :buen_vino_pred, :confianza, :modelo_version)
    """)
    params = {
        'timestamp': datetime.now().isoformat(),
        **datos_entrada,
        **prediccion
    }
    try:
        with engine.begin() as conn:  # commit automatico all'uscita
            resultado = conn.execute(sql, params)
            return resultado.lastrowid
    except Exception as e:
        print(f"Errore nell'inserimento della predizione: {e}")
        return None

# Test
nuevo_id = log_prediccion(
    engine,
    datos_entrada={'tipo_vino': 'blanco', 'alcohol': 12.5, 'pH': 3.2},
    prediccion={'quality_real': 7, 'quality_pred': 6.8,
                'buen_vino_pred': 1, 'confianza': 0.82, 'modelo_version': 'v1.0'}
)
print(f"Predizione registrata con id={nuevo_id}")
```
</details>

---
### Esercizio 5 (sfida) — Dashboard SQL in pandas

**Consegna:** Crea una dashboard testuale con 5 metriche del DB:

1. Totale dei vini per tipo
2. Distribuzione delle qualità (istogramma testuale: `★★★★` = proporzione)
3. Top 3 feature più correlate con `quality` (usa SQL per calcolarle)
4. Confronto delle medie tra vini buoni e non buoni
5. Ultime 5 predizioni registrate

In [ ]:
# ✏️ IL TUO CODICE QUI


---
## Riepilogo

```python
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.orm import DeclarativeBase, Session

# sqlite3 puro
conn   = sqlite3.connect('data/db/mi_bd.db')
cursor = conn.cursor()
cursor.execute("CREATE TABLE IF NOT EXISTS tabla (id INTEGER PRIMARY KEY, valor REAL)")
cursor.executemany("INSERT INTO tabla VALUES (NULL, ?)", [(1.0,), (2.0,)])
conn.commit()
conn.close()

# SQLAlchemy
engine = create_engine('sqlite:///data/db/mi_bd.db')
with engine.connect() as c:
    rows = c.execute(text("SELECT * FROM tabla")).fetchall()

# pandas ↔ SQL
df = pd.read_sql("SELECT * FROM tabla WHERE valor > 1", engine)
df.to_sql('tabla_nueva', engine, if_exists='replace', index=False)
```

## Cosa vedremo nella prossima lezione?
Nella **Lezione 4** addestreremo i primi modelli di Machine Learning con **Scikit-Learn**: da una baseline fino a un RandomForest con GridSearchCV, usando i dati già archiviati nel DB.